# Lecture 2 – Data 100, Spring 2026

[Acknowledgments Page](https://ds100.org/sp26/acks/)

A high-level overview of the [`Polars`](https://pola.rs) library to accompany Lecture 2.

In [1]:
# `pl` is the conventional alias for Polars, as `np` is for NumPy
import polars as pl

## Series and DataFrames

Series and DataFrames are the fundamental `Polars` data structures for storing tabular data and processing the data using vectorized operations.

### Series

A `Series` is a 1-D array of data with a name and a single data type. We can think of it as columnar data.

Let's create a `Series` object and look at two of its components: 1) values and 2) name.

In [2]:
s = pl.Series(["welcome", "to", "data 100"])
s

""
str
"""welcome"""
"""to"""
"""data 100"""


In [3]:
s.to_list()

['welcome', 'to', 'data 100']

In [4]:
# a Series created without a name gets the empty name
s.name

''

In the example above, `Polars` printed the data type of the values alongside them. We can also create a `Series` object by providing a name of our own.

In [5]:
s = pl.Series("my_series", [-1, 10, 2])
s

my_series
i64
-1
10
2


In [6]:
s.to_list()

[-1, 10, 2]

In [7]:
s.name

'my_series'

After it has been created, we can give a `Series` a new name with `rename`.

In [8]:
s = s.rename("renamed_series")
s

renamed_series
i64
-1
10
2


#### Selection in Series
We can select a single value or a set of values in a `Series` using:
- A single position
- A list of positions
- A filtering condition using a **Boolean mask**

In [9]:
s = pl.Series("s", [4, -2, 0, 6])
s

s
i64
4
-2
0
6


**Selection using one or more position(s)**

In [10]:
# Selection using a single position, counting from 0
# Notice how the return value is a single array element
s[0]

4

In [11]:
# Selection using a list of positions
# Notice how the return value is another Series
s[[0, 2]]

s
i64
4
0


**Selection using a filter condition**

In [12]:
# Boolean mask: true for all elements greater than 0.
s > 0

s
bool
true
false
false
true


In [13]:
# Hand the Boolean mask to filter to select data from the original Series: this picks all entries greater than 0
s.filter(s > 0)

s
i64
4
6


<br><br><br><br><br><br><br>
**Instructor Note: Return to slides!**
<br><br><br><br><br><br><br>

#### DataFrame

A `DataFrame` is a 2-D tabular data structure with named columns, where each row is identified by its position in the table. In this lecture, we will see how a `DataFrame` can be created from scratch or loaded from a file. We'll cover the following:
1. From a CSV file
2. From a list of rows
3. From a dictionary of columns
4. From a `Series`

(but there are many more!)

##### Creating a `DataFrame` from a CSV file
For loading data into a `DataFrame`, `Polars` has a number of very useful file reading tools. We'll be using `read_csv` today to load data from a CSV file into a `DataFrame` object.

In [14]:
elections = pl.read_csv("data/elections.csv")
elections

Year,Candidate,Party,Popular vote,Result,%
i64,str,str,i64,str,f64
1824,"""Andrew Jackson""","""Democratic-Republican""",151271,"""loss""",57.210122
1824,"""John Quincy Adams""","""Democratic-Republican""",113142,"""win""",42.789878
1828,"""Andrew Jackson""","""Democratic""",642806,"""win""",56.203927
1828,"""John Quincy Adams""","""National Republican""",500897,"""loss""",43.796073
1832,"""Andrew Jackson""","""Democratic""",702735,"""win""",54.574789
…,…,…,…,…,…
2024,"""Donald Trump""","""Republican""",77303568,"""win""",49.808629
2024,"""Kamala Harris""","""Democratic""",75019230,"""loss""",48.336772
2024,"""Jill Stein""","""Green""",861155,"""loss""",0.554864


`read_csv` takes optional arguments that shape the table as it is read.

In [15]:
# `columns` keeps just the columns we name, in the order we name them
elections = pl.read_csv("data/elections.csv", columns=["Candidate", "Year", "%"])
elections

Year,Candidate,%
i64,str,f64
1824,"""Andrew Jackson""",57.210122
1824,"""John Quincy Adams""",42.789878
1828,"""Andrew Jackson""",56.203927
1828,"""John Quincy Adams""",43.796073
1832,"""Andrew Jackson""",54.574789
…,…,…
2024,"""Donald Trump""",49.808629
2024,"""Kamala Harris""",48.336772
2024,"""Jill Stein""",0.554864


In [16]:
# `n_rows` stops reading after the first few rows, which is handy for a very large file
elections = pl.read_csv("data/elections.csv", n_rows=5)
elections

Year,Candidate,Party,Popular vote,Result,%
i64,str,str,i64,str,f64
1824,"""Andrew Jackson""","""Democratic-Republican""",151271,"""loss""",57.210122
1824,"""John Quincy Adams""","""Democratic-Republican""",113142,"""win""",42.789878
1828,"""Andrew Jackson""","""Democratic""",642806,"""win""",56.203927
1828,"""John Quincy Adams""","""National Republican""",500897,"""loss""",43.796073
1832,"""Andrew Jackson""","""Democratic""",702735,"""win""",54.574789


##### Creating a `DataFrame` from a list of rows

In [17]:
# Creating a DataFrame from a list of rows
# Each row is a list, we specify column names, and orient="row" says to read each list as a row
df_list_1 = pl.DataFrame(
    [["Kiwi", 5.49],
     ["Orange", 3.99]],
    schema=["Fruit", "Price"], orient="row"
)
df_list_1

Fruit,Price
str,f64
"""Kiwi""",5.49
"""Orange""",3.99


In [18]:
# Creating a DataFrame from a list of rows
# Each row is a dictionary
df_list_2 = pl.DataFrame(
    [{"Fruit": "Kiwi", "Price": 5.49},
     {"Fruit": "Orange", "Price": 3.99}]
)
df_list_2

Fruit,Price
str,f64
"""Kiwi""",5.49
"""Orange""",3.99


##### Creating a `DataFrame` from a dictionary of columns

In [19]:
# Creating a DataFrame from a dictionary of columns
# where each columns is a *list*:
df_dict_1 = pl.DataFrame(
    {"Fruit": ["Kiwi", "Orange"],
     "Price": [5.49, 3.99]}
)
df_dict_1

Fruit,Price
str,f64
"""Kiwi""",5.49
"""Orange""",3.99


In [20]:
# first, make some Series
ser_a = pl.Series("ser_a", ["a1", "a2", "a3"])
ser_b = pl.Series("ser_b", ["b1", "b2", "b3"])
ser_a

ser_a
str
"""a1"""
"""a2"""
"""a3"""


In [21]:

# Creating a DataFrame from a dictionary of columns
# where each column is a *Series* (must have matching lengths)
df_dict_ser = pl.DataFrame(
    {"ColumnA": ser_a, "ColumnB": ser_b}
)
df_dict_ser

ColumnA,ColumnB
str,str
"""a1""","""b1"""
"""a2""","""b2"""
"""a3""","""b3"""


##### Creating a `DataFrame` from a `Series`

In [22]:
# Passing a Series to the DataFrame constructor to make a one-column DataFrame
# The Series name becomes the column name
df_ser = pl.DataFrame(ser_a)
df_ser

ser_a
str
"""a1"""
"""a2"""
"""a3"""


In [23]:
# Using to_frame() to convert a Series to DataFrame
ser_to_df = ser_a.to_frame()
ser_to_df

ser_a
str
"""a1"""
"""a2"""
"""a3"""


#### `DataFrame` attributes: `columns`, `dtypes`, and `shape`

In [24]:
# Creating a DataFrame from a CSV file
elections = pl.read_csv("data/elections.csv")
elections

Year,Candidate,Party,Popular vote,Result,%
i64,str,str,i64,str,f64
1824,"""Andrew Jackson""","""Democratic-Republican""",151271,"""loss""",57.210122
1824,"""John Quincy Adams""","""Democratic-Republican""",113142,"""win""",42.789878
1828,"""Andrew Jackson""","""Democratic""",642806,"""win""",56.203927
1828,"""John Quincy Adams""","""National Republican""",500897,"""loss""",43.796073
1832,"""Andrew Jackson""","""Democratic""",702735,"""win""",54.574789
…,…,…,…,…,…
2024,"""Donald Trump""","""Republican""",77303568,"""win""",49.808629
2024,"""Kamala Harris""","""Democratic""",75019230,"""loss""",48.336772
2024,"""Jill Stein""","""Green""",861155,"""loss""",0.554864


In [25]:
elections.dtypes

[Int64, String, String, Int64, String, Float64]

In [26]:
elections.columns

['Year', 'Candidate', 'Party', 'Popular vote', 'Result', '%']

In [27]:
elections.shape

(187, 6)

<br><br><br><br><br><br><br>
**Instructor Note: Return to slides!**
<br><br><br><br><br><br><br>

### Extracting data from `DataFrame`s

In [28]:
elections = pl.read_csv('data/elections.csv')
elections

Year,Candidate,Party,Popular vote,Result,%
i64,str,str,i64,str,f64
1824,"""Andrew Jackson""","""Democratic-Republican""",151271,"""loss""",57.210122
1824,"""John Quincy Adams""","""Democratic-Republican""",113142,"""win""",42.789878
1828,"""Andrew Jackson""","""Democratic""",642806,"""win""",56.203927
1828,"""John Quincy Adams""","""National Republican""",500897,"""loss""",43.796073
1832,"""Andrew Jackson""","""Democratic""",702735,"""win""",54.574789
…,…,…,…,…,…
2024,"""Donald Trump""","""Republican""",77303568,"""win""",49.808629
2024,"""Kamala Harris""","""Democratic""",75019230,"""loss""",48.336772
2024,"""Jill Stein""","""Green""",861155,"""loss""",0.554864


We can use `.head` to return only a few rows of a dataframe.

In [29]:
# By default, calling .head with no argument will show the first 5 rows
elections.head()

Year,Candidate,Party,Popular vote,Result,%
i64,str,str,i64,str,f64
1824,"""Andrew Jackson""","""Democratic-Republican""",151271,"""loss""",57.210122
1824,"""John Quincy Adams""","""Democratic-Republican""",113142,"""win""",42.789878
1828,"""Andrew Jackson""","""Democratic""",642806,"""win""",56.203927
1828,"""John Quincy Adams""","""National Republican""",500897,"""loss""",43.796073
1832,"""Andrew Jackson""","""Democratic""",702735,"""win""",54.574789


In [30]:
elections.head(3)

Year,Candidate,Party,Popular vote,Result,%
i64,str,str,i64,str,f64
1824,"""Andrew Jackson""","""Democratic-Republican""",151271,"""loss""",57.210122
1824,"""John Quincy Adams""","""Democratic-Republican""",113142,"""win""",42.789878
1828,"""Andrew Jackson""","""Democratic""",642806,"""win""",56.203927


We can also use `.tail` to get the last so many rows.

In [31]:
elections.tail(5)

Year,Candidate,Party,Popular vote,Result,%
i64,str,str,i64,str,f64
2024,"""Donald Trump""","""Republican""",77303568,"""win""",49.808629
2024,"""Kamala Harris""","""Democratic""",75019230,"""loss""",48.336772
2024,"""Jill Stein""","""Green""",861155,"""loss""",0.554864
2024,"""Robert Kennedy""","""Independent""",756383,"""loss""",0.487357
2024,"""Chase Oliver""","""Libertarian Party""",650130,"""loss""",0.418895


*What might be some issues with using `.head` or `.tail` to check a DataFrame?*

#### Extraction Using `[]`

`[]` takes up to two arguments, one for row positions and one for column labels. Each argument to `[]` can be:
1. A single value.
2. A list.
3. A slice (a slice of row positions is **exclusive** of its right-hand side, just like normal Python indexing, while a slice of column labels is **inclusive** of both ends).

`[]` selects rows by *position* and columns by *label*.

In [32]:
# Selection by a row position and a column label
elections[0, "Candidate"]

'Andrew Jackson'

In [33]:
# Selection by a list
elections[[87, 25, 179], ["Year", "Party", "%"]]

Year,Party,%
i64,str,f64
1932,"""Republican""",39.830594
1860,"""Southern Democratic""",18.138998
2020,"""Republican""",46.858542


In [34]:
# Selection by a list and a slice of columns
elections[[87, 25, 179], "Popular vote":"%"]

Popular vote,Result,%
i64,str,f64
15761254,"""loss""",39.830594
848019,"""loss""",18.138998
74216154,"""loss""",46.858542


In [35]:
# Extracting all rows using a colon
elections[:, ["Year", "Candidate", "Result"]]

Year,Candidate,Result
i64,str,str
1824,"""Andrew Jackson""","""loss"""
1824,"""John Quincy Adams""","""win"""
1828,"""Andrew Jackson""","""win"""
1828,"""John Quincy Adams""","""loss"""
1832,"""Andrew Jackson""","""win"""
…,…,…
2024,"""Donald Trump""","""win"""
2024,"""Kamala Harris""","""loss"""
2024,"""Jill Stein""","""loss"""


In [36]:
# Extracting all columns using a colon
elections[[87, 25, 179], :]

Year,Candidate,Party,Popular vote,Result,%
i64,str,str,i64,str,f64
1932,"""Herbert Hoover""","""Republican""",15761254,"""loss""",39.830594
1860,"""John C. Breckinridge""","""Southern Democratic""",848019,"""loss""",18.138998
2020,"""Donald Trump""","""Republican""",74216154,"""loss""",46.858542


In [37]:
# Selection by a list and a single-column label
elections[[87, 25, 179], "Popular vote"]

Popular vote
i64
15761254
848019
74216154


In [38]:
# Note that if we pass "Popular vote" in a list, the output will be a DataFrame
elections[[87, 25, 179], ["Popular vote"]]

Popular vote
i64
15761254
848019
74216154


In [39]:
# A condition describes the rows we want without naming their positions, and that is
# the job of filter. We'll see much more of it below.
elections.filter(pl.col("Year") == 2008).select(["Year", "Candidate"])

Year,Candidate
i64,str
2008,"""Barack Obama"""
2008,"""Bob Barr"""
2008,"""Chuck Baldwin"""
2008,"""Cynthia McKinney"""
2008,"""John McCain"""
2008,"""Ralph Nader"""


In [40]:
# If we only use one argument for [], and that argument is a list of integers or a
# slice, Polars uses it for the rows and returns all columns
elections[[180, 181]]

Year,Candidate,Party,Popular vote,Result,%
i64,str,str,i64,str,f64
2020,"""Jo Jorgensen""","""Libertarian""",1865724,"""loss""",1.1779795
2020,"""Howard Hawkins""","""Green""",405035,"""loss""",0.255731


<br><br><br><br><br><br><br>
**Instructor Note: Return to slides!**
<br><br><br><br><br><br><br>

#### Selecting Columns by Position

The second argument to `[]` accepts **column numbers** as well as column labels. The numbers count from the left edge of the table, starting at 0, so `elections[:, 1]` and `elections[:, "Candidate"]` give the same column.

Slicing by column number, like slicing by row position, is **exclusive** of the right-hand side of the slice. The inclusive behavior above belongs to label slices only.

A row's position is the only handle you have on it, so any operation that reorders the table hands out new positions. We come back to that at the end of the lecture.

In [41]:
# Extracting the value at the first row (row 0) and the second column.
# Remember that Python indexing begins at position 0!
elections[0, 1]

'Andrew Jackson'

In [42]:
# Extracting the second, third, and fourth rows of the second column
# (returns a series since we used a single column)
elections[[1, 2, 3], 1]

Candidate
str
"""John Quincy Adams"""
"""Andrew Jackson"""
"""John Quincy Adams"""


In [43]:
# Select the rows at positions 1, 2, and 3.
# Select the columns at positions 0, 1, and 2.
elections[[1, 2, 3], [0, 1, 2]]

Year,Candidate,Party
i64,str,str
1824,"""John Quincy Adams""","""Democratic-Republican"""
1828,"""Andrew Jackson""","""Democratic"""
1828,"""John Quincy Adams""","""National Republican"""


In [44]:
# Position-based extraction using a list of rows and a slice of column numbers
elections[[1, 2, 3], 0:3]

Year,Candidate,Party
i64,str,str
1824,"""John Quincy Adams""","""Democratic-Republican"""
1828,"""Andrew Jackson""","""Democratic"""
1828,"""John Quincy Adams""","""National Republican"""


In [45]:
# Selecting all rows using a colon
elections[:, 0:3]

Year,Candidate,Party
i64,str,str
1824,"""Andrew Jackson""","""Democratic-Republican"""
1824,"""John Quincy Adams""","""Democratic-Republican"""
1828,"""Andrew Jackson""","""Democratic"""
1828,"""John Quincy Adams""","""National Republican"""
1832,"""Andrew Jackson""","""Democratic"""
…,…,…
2024,"""Donald Trump""","""Republican"""
2024,"""Kamala Harris""","""Democratic"""
2024,"""Jill Stein""","""Green"""


In [46]:
# If we only use one argument for [], Polars uses it
# for the rows, and returns all columns
elections[138:144]

Year,Candidate,Party,Popular vote,Result,%
i64,str,str,i64,str,f64
1988,"""Ron Paul""","""Libertarian""",431750,"""loss""",0.47266
1992,"""Andre Marrou""","""Libertarian""",290087,"""loss""",0.278516
1992,"""Bill Clinton""","""Democratic""",44909806,"""win""",43.118485
1992,"""Bo Gritz""","""Populist""",106152,"""loss""",0.101918
1992,"""George H. W. Bush""","""Republican""",39104550,"""loss""",37.544784
1992,"""Ross Perot""","""Independent""",19743821,"""loss""",18.956298


<br><br><br><br><br><br><br>
**Instructor Note: Return to slides!**
<br><br><br><br><br><br><br>

#### Extraction using `filter` and `select`

`[]` asks for positions and labels, and it yields concise code for the most common quick looks at a table. Anything *computed*, though (a condition, an arithmetic result), goes through a second pair of methods.

`filter` chooses rows and `select` chooses columns. Both take **expressions**, which are built with `pl.col` and can compare and combine columns before anything is returned. This is the pairing you'll reach for most often, because a condition like "more than 60 million popular votes" describes the rows you want without needing to know where they sit.

Two rules cover most of the confusion: `filter` narrows the rows and leaves every column in place, and `select` decides which columns come back.

In [47]:
# If we provide a single column name to [], we get back that column as a Series
elections["Candidate"]

Candidate
str
"""Andrew Jackson"""
"""John Quincy Adams"""
"""Andrew Jackson"""
"""John Quincy Adams"""
"""Andrew Jackson"""
…
"""Donald Trump"""
"""Kamala Harris"""
"""Jill Stein"""


In [48]:
# select takes a list of column names and returns a DataFrame
elections.select(["Year", "Candidate", "Result"])

Year,Candidate,Result
i64,str,str
1824,"""Andrew Jackson""","""loss"""
1824,"""John Quincy Adams""","""win"""
1828,"""Andrew Jackson""","""win"""
1828,"""John Quincy Adams""","""loss"""
1832,"""Andrew Jackson""","""win"""
…,…,…
2024,"""Donald Trump""","""win"""
2024,"""Kamala Harris""","""loss"""
2024,"""Jill Stein""","""loss"""


In [49]:
# select also accepts a computed expression, and .alias names the result
elections.select((pl.col("Popular vote") / 1_000_000).alias("Popular vote (millions)"))

Popular vote (millions)
f64
0.151271
0.113142
0.642806
0.500897
0.702735
…
77.303568
75.01923
0.861155


In [50]:
# filter takes a condition and returns the rows that satisfy it
elections.filter(pl.col("Popular vote") > 60000000)

Year,Candidate,Party,Popular vote,Result,%
i64,str,str,i64,str,f64
2004,"""George W. Bush""","""Republican""",62040610,"""win""",50.771824
2008,"""Barack Obama""","""Democratic""",69498516,"""win""",53.02351
2012,"""Barack Obama""","""Democratic""",65915795,"""win""",51.258484
2012,"""Mitt Romney""","""Republican""",60933504,"""loss""",47.384076
2016,"""Donald Trump""","""Republican""",62984828,"""win""",46.407862
2016,"""Hillary Clinton""","""Democratic""",65853514,"""loss""",48.521539
2020,"""Joseph Biden""","""Democratic""",81268924,"""win""",51.311515
2020,"""Donald Trump""","""Republican""",74216154,"""loss""",46.858542
2024,"""Donald Trump""","""Republican""",77303568,"""win""",49.808629


<br><br><br><br><br><br><br>
**Instructor Note: Return to slides!**
<br><br><br><br><br><br><br>

#### Boolean Operators

To filter on multiple conditions, we combine boolean masks using **bitwise comparisons**.

Symbol | Usage      | Meaning
------ | ---------- | -------------------------------------
~    | ~p       | Returns negation of p
&#124; | p &#124; q | p OR q
&    | p & q    | p AND q
^  | p ^ q | p XOR q (exclusive or)

**Always** make sure you wrap each condition inside `()` when combining boolean masks with bitwise comparisons!

In [51]:
# Grab rows from 2008 OR candidates winning over 60% of the vote (or both)
elections.filter((pl.col("Year") == 2008) | (pl.col("%") >= 60))

Year,Candidate,Party,Popular vote,Result,%
i64,str,str,i64,str,f64
1920,"""Warren Harding""","""Republican""",16144093,"""win""",60.574501
1936,"""Franklin Roosevelt""","""Democratic""",27752648,"""win""",60.978107
1964,"""Lyndon Johnson""","""Democratic""",43127041,"""win""",61.344703
1972,"""Richard Nixon""","""Republican""",47168710,"""win""",60.907806
2008,"""Barack Obama""","""Democratic""",69498516,"""win""",53.02351
2008,"""Bob Barr""","""Libertarian""",523715,"""loss""",0.399565
2008,"""Chuck Baldwin""","""Constitution""",199750,"""loss""",0.152398
2008,"""Cynthia McKinney""","""Green""",161797,"""loss""",0.123442
2008,"""John McCain""","""Republican""",59948323,"""loss""",45.737243


In [52]:
# Don't do this! Python's precedence rules (like PEMDAS for other operators) will do the wrong thing
# elections.filter(pl.col("Year") == 2008 | pl.col("%") >= 60)

In [53]:
# Grab post-2000 winners: rows where year is after 2000 AND result is win
elections.filter((pl.col("Year") > 2000) & (pl.col("Result") == "win"))

Year,Candidate,Party,Popular vote,Result,%
i64,str,str,i64,str,f64
2004,"""George W. Bush""","""Republican""",62040610,"""win""",50.771824
2008,"""Barack Obama""","""Democratic""",69498516,"""win""",53.02351
2012,"""Barack Obama""","""Democratic""",65915795,"""win""",51.258484
2016,"""Donald Trump""","""Republican""",62984828,"""win""",46.407862
2020,"""Joseph Biden""","""Democratic""",81268924,"""win""",51.311515
2024,"""Donald Trump""","""Republican""",77303568,"""win""",49.808629


In [54]:
# To make code more readable, use multiple lines
elections.filter(
    (pl.col("Year") < 2000) &
    (pl.col("Year") > 1941) &
    (pl.col("Result") == "win") &
    (pl.col("%") >= 55)
)

Year,Candidate,Party,Popular vote,Result,%
i64,str,str,i64,str,f64
1952,"""Dwight Eisenhower""","""Republican""",34075529,"""win""",55.325173
1956,"""Dwight Eisenhower""","""Republican""",35579180,"""win""",57.650654
1964,"""Lyndon Johnson""","""Democratic""",43127041,"""win""",61.344703
1972,"""Richard Nixon""","""Republican""",47168710,"""win""",60.907806
1984,"""Ronald Reagan""","""Republican""",54455472,"""win""",59.023326


#### Preparing the data for our graph

![A graph from 1900-2020 (x axis) and approximately 20% to 60% (y axis) shows the party preference as lines changing over time. Democratic preference is in blue and Republican is in red. There is no clear preference over time, however since approx 1990, there is less of a gap between the parties. On the right is an annotated dataframe with the year ("After 1900 only"), party ("Dem/Rep only"), and %. We see two rows in the dataframe for each year. ](images/graph_and_df.png)

We need rows that are either Democratic OR Republican, AND after 1900. We also need only the Year, Party, and % columns.

In [55]:
# Notice the parens: it's important to use ((dem | rep) & year) so that we
# get the boolean operations right
elections.filter(
    ((pl.col('Party') == 'Democratic') | (pl.col('Party') == 'Republican')) &
    (pl.col('Year') > 1900)
).select(['Year', 'Party', '%'])

Year,Party,%
i64,str,f64
1904,"""Democratic""",37.685116
1904,"""Republican""",56.562787
1908,"""Democratic""",43.41464
1908,"""Republican""",52.0133
1912,"""Republican""",23.218466
…,…,…
2016,"""Democratic""",48.521539
2020,"""Democratic""",51.311515
2020,"""Republican""",46.858542


##### Investigating 1992

In [56]:
# What happened in 1992?
elections.filter(pl.col('Year') == 1992)

Year,Candidate,Party,Popular vote,Result,%
i64,str,str,i64,str,f64
1992,"""Andre Marrou""","""Libertarian""",290087,"""loss""",0.278516
1992,"""Bill Clinton""","""Democratic""",44909806,"""win""",43.118485
1992,"""Bo Gritz""","""Populist""",106152,"""loss""",0.101918
1992,"""George H. W. Bush""","""Republican""",39104550,"""loss""",37.544784
1992,"""Ross Perot""","""Independent""",19743821,"""loss""",18.956298


In [57]:
# How do we put that in context? How does 19% compare to other third party candidates?
third_party = elections.filter(
    (pl.col('Party') != "Democratic") &
    (pl.col('Party') != "Republican") &
    (pl.col('Year') > 1900)
)
third_party.sort('%').tail(15)

Year,Candidate,Party,Popular vote,Result,%
i64,str,str,i64,str,f64
1948,"""Henry A. Wallace""","""Progressive""",1157328,"""loss""",2.374144
1948,"""Strom Thurmond""","""Dixiecrat""",1175930,"""loss""",2.412304
2000,"""Ralph Nader""","""Green""",2882955,"""loss""",2.741176
1908,"""Eugene V. Debs""","""Socialist""",420852,"""loss""",2.850866
1904,"""Eugene V. Debs""","""Socialist""",402810,"""loss""",2.985897
…,…,…,…,…,…
1996,"""Ross Perot""","""Reform""",8085294,"""loss""",8.408844
1968,"""George Wallace""","""American Independent""",9901118,"""loss""",13.571218
1924,"""Robert La Follette""","""Progressive""",4831706,"""loss""",16.694596


#### Working with row positions

A row is identified by its position in the table, counting from 0. Those positions are not stored anywhere: `with_row_index` writes them into a column of their own when we want to keep them.

In [58]:
# Creating a DataFrame from a CSV file
elections = pl.read_csv("data/elections.csv")
elections.head(3)

Year,Candidate,Party,Popular vote,Result,%
i64,str,str,i64,str,f64
1824,"""Andrew Jackson""","""Democratic-Republican""",151271,"""loss""",57.210122
1824,"""John Quincy Adams""","""Democratic-Republican""",113142,"""win""",42.789878
1828,"""Andrew Jackson""","""Democratic""",642806,"""win""",56.203927


Sorting the table hands out new positions, so record the old ones *before* sorting if you want them afterwards.

In [59]:
# with_row_index adds a column holding each row's current position
elections.with_row_index("original_position").sort("%", descending=True).head()

original_position,Year,Candidate,Party,Popular vote,Result,%
u32,i64,str,str,i64,str,f64
114,1964,"""Lyndon Johnson""","""Democratic""",43127041,"""win""",61.344703
91,1936,"""Franklin Roosevelt""","""Democratic""",27752648,"""win""",60.978107
120,1972,"""Richard Nixon""","""Republican""",47168710,"""win""",60.907806
79,1920,"""Warren Harding""","""Republican""",16144093,"""win""",60.574501
133,1984,"""Ronald Reagan""","""Republican""",54455472,"""win""",59.023326


In [60]:
# Notice that with_row_index, just like most DataFrame methods, returns a new dataframe (instead of modifying it)
elections

Year,Candidate,Party,Popular vote,Result,%
i64,str,str,i64,str,f64
1824,"""Andrew Jackson""","""Democratic-Republican""",151271,"""loss""",57.210122
1824,"""John Quincy Adams""","""Democratic-Republican""",113142,"""win""",42.789878
1828,"""Andrew Jackson""","""Democratic""",642806,"""win""",56.203927
1828,"""John Quincy Adams""","""National Republican""",500897,"""loss""",43.796073
1832,"""Andrew Jackson""","""Democratic""",702735,"""win""",54.574789
…,…,…,…,…,…
2024,"""Donald Trump""","""Republican""",77303568,"""win""",49.808629
2024,"""Kamala Harris""","""Democratic""",75019230,"""loss""",48.336772
2024,"""Jill Stein""","""Green""",861155,"""loss""",0.554864


In [61]:
# Adding the index column after the sort numbers the rows in their new order instead
elections.sort("%", descending=True).with_row_index().head()

index,Year,Candidate,Party,Popular vote,Result,%
u32,i64,str,str,i64,str,f64
0,1964,"""Lyndon Johnson""","""Democratic""",43127041,"""win""",61.344703
1,1936,"""Franklin Roosevelt""","""Democratic""",27752648,"""win""",60.978107
2,1972,"""Richard Nixon""","""Republican""",47168710,"""win""",60.907806
3,1920,"""Warren Harding""","""Republican""",16144093,"""win""",60.574501
4,1984,"""Ronald Reagan""","""Republican""",54455472,"""win""",59.023326


## Slido Exercises

**Question 1**

What's the output of the following code?

In [62]:
example = pl.Series(
    "example",
    [4, 5, 6]
)
example.filter(example > 4).to_list()

[5, 6]

**Questions 2, 3, and 4**

Which of the following statements return the string `"blue fish"` itself, rather than a `Series` or `DataFrame` that contains it?

In [63]:
weird = pl.DataFrame({"a":["one fish", "two fish"],
                      "b":["red fish", "blue fish"]})
weird

a,b
str,str
"""one fish""","""red fish"""
"""two fish""","""blue fish"""


In [64]:
# weird[1, 'b']

In [65]:
weird

a,b
str,str
"""one fish""","""red fish"""
"""two fish""","""blue fish"""


In [66]:
# weird[0, 'b']

In [67]:
weird

a,b
str,str
"""one fish""","""red fish"""
"""two fish""","""blue fish"""


In [68]:
# weird[['b', 1]]

In [69]:
weird

a,b
str,str
"""one fish""","""red fish"""
"""two fish""","""blue fish"""


In [70]:
# weird['b', 1]

In [71]:
weird

a,b
str,str
"""one fish""","""red fish"""
"""two fish""","""blue fish"""


In [72]:
# weird[1, 1]

In [73]:
weird

a,b
str,str
"""one fish""","""red fish"""
"""two fish""","""blue fish"""


In [74]:
# weird[1, :]

In [75]:
weird

a,b
str,str
"""one fish""","""red fish"""
"""two fish""","""blue fish"""


In [76]:
# weird[2, 2]

In [77]:
weird

a,b
str,str
"""one fish""","""red fish"""
"""two fish""","""blue fish"""


In [78]:
# weird[[1, 1]]

In [79]:
weird

a,b
str,str
"""one fish""","""red fish"""
"""two fish""","""blue fish"""


In [80]:
# weird['b'][1]

In [81]:
weird

a,b
str,str
"""one fish""","""red fish"""
"""two fish""","""blue fish"""


In [82]:
# weird[1]['b']

In [83]:
weird

a,b
str,str
"""one fish""","""red fish"""
"""two fish""","""blue fish"""


In [84]:
# weird.select('b')[1]

In [85]:
weird

a,b
str,str
"""one fish""","""red fish"""
"""two fish""","""blue fish"""


In [86]:
# weird.row(1)